In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

dbutils.widgets.removeAll()

## PARAMETRIZAR CATALOGO A PROD
dbutils.widgets.text("PRM_catalogo","catalogo_desa_intEcommerce")

PRM_catalogo = dbutils.widgets.get("PRM_catalogo")

def read_tablaCliente():
    df_cliente = spark.table(f"{PRM_catalogo}.bronze.Clientes_Sistema") \
        .select(
            trim(col("User_id")).alias("ID_Cliente"), 
            trim(col("Last_Name")).alias("Apellido"), 
            trim(col("Name")).alias("Nombre"),  
            col("Age").cast("int").alias("Edad"),
            when(
                (col("Cell_number").isNull()) | 
                (trim(col("Cell_number")) == "") | 
                (col("Cell_number") == "-"), 
                "CLI_SNCEL"
            ).when(
                (substring(col("Cell_number"), 1, 1) != "9") | 
                (length(col("Cell_number")) != 9), 
                "NUM_INVAL"
            ).otherwise(col("Cell_number")).alias("Numero_Celular"),
            to_date(col("Sign_date")).alias("Fecha_inscripcion_PagWeb"),                      
            to_timestamp(date_format(current_timestamp(), "yyyy-MM-dd")).alias("Fecha_proceso")             
        )    
    return df_cliente

def read_tablaProductos():
    df_productos = spark.table(f"{PRM_catalogo}.bronze.Productos_Sistema") \
        .select(
            trim(col("Product_id")).alias("Cod_producto"),  
            trim(col("Producto_Name")).alias("Nombre_producto"),  
            trim(col("Category")).alias("Categoria"), 
            trim(col("Price")).cast(DecimalType(16,2)).alias("Precio"),
            to_timestamp(date_format(current_timestamp(), "yyyy-MM-dd")).alias("Fecha_proceso")    
        )    
    return df_productos

def read_tablatipoInteraccion():
    df_tipinteraccion = spark.table(f"{PRM_catalogo}.bronze.Interaccion_Sistema") \
        .select(
            trim(col("TypeInt_id")).alias("Cod_Tipo_Interaccion"),  
            trim(col("Interaction_name")).alias("Nombre_codigo_sistema"), 
            trim(col("DestipoInt_Spanish")).alias("Descripcion_interaccion"),
            to_date(col("Registration_system_date"),"dd/mm/yyyy").alias("Fecha_registro_sistema"),
            to_timestamp(date_format(current_timestamp(), "yyyy-MM-dd")).alias("Fecha_proceso")      
        )    
    return df_tipinteraccion

def Cleancliente(df_cliente):
    return df_cliente.where(
        col("ID_Cliente").isNotNull() & col("Apellido").isNotNull() & col("Nombre").isNotNull()
    ).dropDuplicates(["ID_Cliente", "Apellido", "Nombre"])

def Cleanproducto(df_productos):
    return df_productos.where(
        col("Cod_producto").isNotNull() & col("Nombre_producto").isNotNull() & col("Precio").isNotNull()
    ).dropDuplicates(["Cod_producto", "Nombre_producto", "Precio"])

def Cleantipoint(df_tipinteraccion):
    return df_tipinteraccion.where(
        col("Cod_Tipo_Interaccion").isNotNull() & col("Nombre_codigo_sistema").isNotNull() & col("Fecha_registro_sistema").isNotNull()
    ).dropDuplicates(["Cod_Tipo_Interaccion", "Nombre_codigo_sistema", "Fecha_registro_sistema"])

def main():
 
    df_cliente = read_tablaCliente()
    df_productos = read_tablaProductos()
    df_tipinteraccion = read_tablatipoInteraccion()

 
    df_cliente_final = Cleancliente(df_cliente)
    df_productos_final = Cleanproducto(df_productos)
    df_tipinteraccion_final = Cleantipoint(df_tipinteraccion) 

 
    df_cliente_final.write.mode("overwrite").saveAsTable(f"{PRM_catalogo}.silver.Tabla_Cliente")
    df_productos_final.write.mode("overwrite").saveAsTable(f"{PRM_catalogo}.silver.Tabla_Producto")
    df_tipinteraccion_final.write.mode("overwrite").saveAsTable(f"{PRM_catalogo}.silver.Tabla_Destipinteraccion")

main()

